In [ ]:
!pip install --upgrade mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!unzip npy_dataset.zip -d ../

Streaming output truncated to the last 5000 lines.
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_24)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_20)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_34_07)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_42)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_41)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_33_49)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_32_57)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_56)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_31)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_34)_c.npy  
  inflating: ../content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_45)_c.npy  
  i

In [ ]:
def gather_tasks(directories, dest_dir):
    """Crawls directories and builds a list of inputs and outputs for the workers."""
    tasks = []

    for d in directories:
        if not os.path.exists(d):
            continue

        classes = [c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c))]

        for class_name in classes:
            class_source_path = os.path.join(d, class_name)
            class_dest_path = os.path.join(dest_dir, class_name)

            os.makedirs(class_dest_path, exist_ok=True)

            videos = glob.glob(os.path.join(class_source_path, '*.mp4'))

            for video_path in videos:
                video_filename = os.path.basename(video_path)
                output_path = os.path.join(class_dest_path, video_filename.replace('.mp4', '.npy'))

                # Only add to task queue if not already processed
                if not os.path.exists(output_path):
                    tasks.append((video_path, output_path))

    return tasks


In [ ]:
import os
import glob
base_path = '/content/drive/MyDrive/arabic sign language'

COLAB_OUT_DIR_train = '/content/npy_dataset/train'
COLAB_OUT_DIR_test = '/content/npy_dataset/test'

train_dirs = [
    os.path.join(base_path, 'extracted_03_train'),
]

test_dirs = [
    os.path.join(base_path, 'extracted_03_test'),
]

print("--- Gathering Training Tasks ---")
train_tasks = gather_tasks(train_dirs, COLAB_OUT_DIR_train)
print(len(train_tasks))

print("\n--- Gathering Testing Tasks ---")
test_tasks = gather_tasks(test_dirs, COLAB_OUT_DIR_test)
print(len(test_tasks))

--- Gathering Training Tasks ---
1842

--- Gathering Testing Tasks ---
4006


In [ ]:
!wget -q -O /content/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import concurrent.futures
from tqdm.notebook import tqdm

# ==========================================
# 1. MEDIAPIPE TASKS SETUP (Worker Function)
# ==========================================
def extract_hands(result):
    """
    Parses the HandLandmarkerResult.
    Maps detected hands to Left and Right arrays, filling missing ones with zeros.
    Total features: (21 LH + 21 RH) * 3 = 126
    """
    lh = np.zeros(21 * 3)
    rh = np.zeros(21 * 3)

    # If no hands detected at all
    if not result or not result.hand_landmarks:
        return np.concatenate([lh, rh])

    # Map the detected hands based on their predicted handedness
    for idx, handedness in enumerate(result.handedness):
        hand_label = handedness[0].category_name # "Left" or "Right"
        landmarks = result.hand_landmarks[idx]

        flat_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in landmarks]).flatten()

        if hand_label == "Left":
            lh = flat_landmarks
        elif hand_label == "Right":
            rh = flat_landmarks

    return np.concatenate([lh, rh])

def process_single_video(video_path, output_path, model_path='hand_landmarker.task'):
    """Processes a single video using the Tasks API. Designed to run in a separate process."""
    try:
        # Create the landmarker instance inside the worker process
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.HandLandmarkerOptions(
            base_options=base_options,
            num_hands=2,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            running_mode=vision.RunningMode.VIDEO
        )

        frames_data = []
        cap = cv2.VideoCapture(video_path)

        # We need the FPS to calculate manual timestamps for the Tasks API
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps == 0 or np.isnan(fps):
            fps = 30.0 # Fallback

        with vision.HandLandmarker.create_from_options(options) as landmarker:
            frame_index = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                if frame_index % 2 != 0:
                  frame_index += 1
                  continue

                frame = cv2.resize(frame, (640, 480))
                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image_rgb.flags.writeable = False

                # Convert to MediaPipe Image object
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                # Calculate strictly increasing timestamp in milliseconds
                timestamp_ms = int(frame_index * (1000 / fps))

                # Process frame
                result = landmarker.detect_for_video(mp_image, timestamp_ms)
                frames_data.append(extract_hands(result))

                frame_index += 1

        cap.release()

        # Save array
        if len(frames_data) > 0:
            np.save(output_path, np.array(frames_data, dtype=np.float16))

        return True, video_path

    except Exception as e:
        return False, f"Error on {os.path.basename(video_path)}: {e}"


# ==========================================
# 2. MULTI-CORE BATCH PROCESSING LOGIC
# ==========================================

def run_parallel_pipeline(tasks):
    """Executes the tasks across all available CPU cores."""
    if not tasks:
        print("No new videos to process.")
        return

    # Use all available CPU cores on the Colab instance
    max_workers = os.cpu_count()
    print(f"Starting ProcessPoolExecutor with {max_workers} cores...")

    # Spawn processes
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks to the executor
        futures = [executor.submit(process_single_video, t[0], t[1]) for t in tasks]

        # Wrap the completion iterator with tqdm for a single, unified progress bar
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc="Processing Videos"):
            success, message = future.result()
            if not success:
                print(message)


# ==========================================
# 3. EXECUTION
# ==========================================

print(f"Found {len(train_tasks)} training videos to process.")
run_parallel_pipeline(train_tasks)

print(f"Found {len(test_tasks)} testing videos to process.")
run_parallel_pipeline(test_tasks)

print("\nExtraction complete! Ready for Kaggle upload.")

Found 1842 training videos to process.
Starting ProcessPoolExecutor with 2 cores...


Processing Videos:   0%|          | 0/1842 [00:00<?, ?it/s]

Found 4006 testing videos to process.
Starting ProcessPoolExecutor with 2 cores...


Processing Videos:   0%|          | 0/4006 [00:00<?, ?it/s]


Extraction complete! Ready for Kaggle upload.


In [ ]:
import os
import json
from google.colab import userdata

# --- CHANGE THIS FOR EACH NOTEBOOK (1, 2, or 3) ---
PART_NUMBER = 3
# -------------------------------------------------

# 1. Setup API Credentials (assuming you added them to Colab Secrets)
!pip install kaggle -q
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

DATASET_PATH = "/content/npy_dataset"

# 2. Generate Unique Metadata for this Part
metadata = {
  "title": f"Arabic Sign Language Landmarks NPY - Part {PART_NUMBER}",
  # ID must be lowercase, alphanumeric, and dashes only
  "id": f"{os.environ['KAGGLE_USERNAME']}/arsl-landmarks-npy-part{PART_NUMBER}",
  "licenses": [{"name": "CC0-1.0"}]
}

with open(os.path.join(DATASET_PATH, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f)

# 3. Authenticate and Upload
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

print(f"Compressing and uploading Part {PART_NUMBER} to Kaggle...")

# Create a brand new dataset for this specific chunk
api.dataset_create_new(
    folder=DATASET_PATH,
    dir_mode='zip',
    convert_to_csv=False,
    public=False
)

print(f"Part {PART_NUMBER} upload successful!")

Compressing and uploading Part 3 to Kaggle...
Starting upload for file test.zip


100%|██████████| 15.4M/15.4M [00:01<00:00, 10.6MB/s]


Upload successful: test.zip (15MB)
Starting upload for file train.zip


100%|██████████| 80.9M/80.9M [00:03<00:00, 27.3MB/s]


Upload successful: train.zip (81MB)
Part 3 upload successful!


In [ ]:
!zip -r npy_dataset.zip /content/npy_dataset

Streaming output truncated to the last 5000 lines.
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_24)_c.npy (deflated 45%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_20)_c.npy (deflated 45%)
  adding: content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_34_07)_c.npy (deflated 40%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_42)_c.npy (deflated 34%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_41)_c.npy (deflated 42%)
  adding: content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_33_49)_c.npy (deflated 36%)
  adding: content/npy_dataset/train/0401/03_03_0401_(30_11_17_21_32_57)_c.npy (deflated 43%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_56)_c.npy (deflated 47%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_14_31)_c.npy (deflated 38%)
  adding: content/npy_dataset/train/0401/03_03_0401_(27_02_18_20_15_34)_c.npy (deflated 39%)
  adding: content/n

In [ ]:
from google.colab import files
files.download("/content/npy_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>